# FDI, Economic Growth, Inflation, and Exchange Rate Volatility
### A Regression Analysis of Iran and Regional Economies

**Author:** Leila Rezaei  
**Research interests:** Applied economics, international business

---

## Research Question
Do GDP growth, inflation, and exchange rate volatility significantly affect Foreign Direct Investment (FDI) net inflows in Iran, compared to regional peer economies (Turkey, Saudi Arabia, UAE)?

## Hypothesis
H1: Higher exchange rate volatility is associated with lower FDI inflows (as % of GDP).  
H2: Higher GDP growth is associated with higher FDI inflows.  
H3: Higher inflation is associated with lower FDI inflows.

## Data Source
World Bank World Development Indicators (WDI), accessed via the `wbdata` Python package, 2000–2023.

## Method
Pooled OLS regression with country fixed effects, using `statsmodels`.

## 1. Setup
Run this cell first. If you are in Google Colab, this installs the one package that isn't pre-installed (`wbdata`).

In [ ]:
!pip install wbdata -q

In [ ]:
import pandas as pd
import numpy as np
import wbdata
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## 2. Fetch Data from the World Bank API

Indicators used:
- `BX.KLT.DINV.WD.GD.ZS` — FDI, net inflows (% of GDP)
- `NY.GDP.MKTP.KD.ZG` — GDP growth (annual %)
- `FP.CPI.TOTL.ZG` — Inflation, consumer prices (annual %)
- `PA.NUS.FCRF` — Official exchange rate (LCU per US$, period average)

Countries: Iran (IRN), Turkey (TUR), Saudi Arabia (SAU), United Arab Emirates (ARE).

In [ ]:
indicators = {
    'BX.KLT.DINV.WD.GD.ZS': 'fdi_pct_gdp',
    'NY.GDP.MKTP.KD.ZG': 'gdp_growth',
    'FP.CPI.TOTL.ZG': 'inflation',
    'PA.NUS.FCRF': 'exchange_rate'
}

countries = ['IRN', 'TUR', 'SAU', 'ARE']

raw = wbdata.get_dataframe(indicators, country=countries, date=('2000', '2023'))
raw = raw.reset_index()
raw.head()

## 3. Clean the Data

In [ ]:
df = raw.rename(columns={'country': 'country'}).copy()
df['date'] = df['date'].astype(int)
df = df.sort_values(['country', 'date']).reset_index(drop=True)

# Year-over-year % change in the exchange rate = our volatility proxy
df['fx_change'] = df.groupby('country')['exchange_rate'].pct_change() * 100

# Drop rows with missing values in the variables we need for the model
model_df = df.dropna(subset=['fdi_pct_gdp', 'gdp_growth', 'inflation', 'fx_change']).copy()

print(f'Rows before cleaning: {len(df)}')
print(f'Rows after cleaning:  {len(model_df)}')
model_df.head()

## 4. Exploratory Data Analysis

In [ ]:
model_df.groupby('country')[['fdi_pct_gdp', 'gdp_growth', 'inflation', 'fx_change']].describe().T

In [ ]:
fig, ax = plt.subplots()
for c, sub in model_df.groupby('country'):
    ax.plot(sub['date'], sub['fdi_pct_gdp'], marker='o', label=c)
ax.set_title('FDI Net Inflows (% of GDP), 2000-2023')
ax.set_xlabel('Year')
ax.set_ylabel('FDI (% of GDP)')
ax.legend()
plt.show()

In [ ]:
corr = model_df[['fdi_pct_gdp', 'gdp_growth', 'inflation', 'fx_change']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.scatterplot(data=model_df, x='gdp_growth', y='fdi_pct_gdp', hue='country', ax=axes[0], legend=False)
axes[0].set_title('FDI vs GDP Growth')
sns.scatterplot(data=model_df, x='inflation', y='fdi_pct_gdp', hue='country', ax=axes[1], legend=False)
axes[1].set_title('FDI vs Inflation')
sns.scatterplot(data=model_df, x='fx_change', y='fdi_pct_gdp', hue='country', ax=axes[2])
axes[2].set_title('FDI vs Exchange Rate Volatility')
plt.tight_layout()
plt.show()

## 5. Regression Model

Pooled OLS with country fixed effects:

$$FDI_{it} = \beta_0 + \beta_1 GDPGrowth_{it} + \beta_2 Inflation_{it} + \beta_3 FXChange_{it} + \text{Country FE} + \varepsilon_{it}$$

In [ ]:
model = smf.ols(
    'fdi_pct_gdp ~ gdp_growth + inflation + fx_change + C(country)',
    data=model_df
).fit(cov_type='HC3')  # robust standard errors

print(model.summary())

## 6. Interpretation

*(Fill this in after running the cells above with the real output.)*

- **GDP growth coefficient:** interpret sign and significance (p < 0.05?) — supports/rejects H2.
- **Inflation coefficient:** interpret sign and significance — supports/rejects H3.
- **Exchange rate volatility (`fx_change`) coefficient:** interpret sign and significance — supports/rejects H1.
- **R-squared:** how much variance in FDI is explained by the model.
- **Country fixed effects:** note whether Iran's baseline FDI differs significantly from the reference country.

## 7. Limitations

- Small sample (4 countries × ~20 years) limits statistical power.
- FDI is affected by many unobserved factors (sanctions, geopolitical events, oil prices) not captured here.
- World Bank data for Iran has gaps in some years due to reporting limitations.
- Correlation/regression here does not establish causation.

## 8. Save Cleaned Data

In [ ]:
model_df.to_csv('../data/wdi_fdi_data.csv', index=False)
print('Saved to data/wdi_fdi_data.csv')